<a href="https://colab.research.google.com/github/Ewanjohndennis/flyrankml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- **Feature window:** Trailing 90 days ending on the last day of the snapshot month.
- **Label window:** 30-day change in impressions (`imp_last30` vs `imp_prev30`).
- **Iteration month:** `2026-03` (mid-panel).
- **Sealed test month:** `2026-06` (final month).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Category | Window / Details |
|---|---|---|
| `imp_prev30` | Feature | Historical days 31–60 before snapshot |
| `days_with_impressions` | Feature | Trailing 90-day count |
| `avg_position` | Feature | Trailing 90-day average |
| `is_declining` | Label | `imp_last30 < 0.8 * imp_prev30` |
| `client_hash_id` | Context | Client-level split key |
| `imp_last30` | Excluded (Feature) | Numerator of label (leaks current window) |

In [2]:
import getpass, os
import duckdb
import pandas as pd

try:
  from google.colab import userdata

  HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
  HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF READ token: ')

con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = 'hf://datasets/FlyRank/internship-warehouse'

DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
FACT_SAMPLE = (
    f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')"
)
FACT_DAILY = (
    f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
)
FACT_QUERY = f"read_parquet('{REL}/fact_content_query_90d.parquet')"

MONTH = '2026-03'

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

# Feature extraction query
feature_frame = con.sql(f"""
    WITH month_data AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date <= DATE_TRUNC('month', DATE '{MONTH}-01') + INTERVAL 30 DAY - INTERVAL 1 DAY
                     AND report_date >= DATE_TRUNC('month', DATE '{MONTH}-01')
                THEN gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN report_date > DATE_TRUNC('month', DATE '{MONTH}-01') + INTERVAL 30 DAY - INTERVAL 1 DAY
                THEN gsc_impressions ELSE 0 END) AS imp_last30,
            COUNT(CASE WHEN gsc_impressions > 0 THEN 1 END) AS days_with_impressions,
            AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position,
            SUM(gsc_clicks) * 1.0 / NULLIF(COUNT(*), 0) AS avg_daily_clicks
        FROM {FACT_DAILY}
        WHERE strftime(report_date, '%Y-%m') = '{MONTH}'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT *,
        CASE WHEN imp_last30 < 0.8 * NULLIF(imp_prev30, 0) THEN 1 ELSE 0 END AS is_declining
    FROM month_data
    WHERE imp_prev30 > 0
""").df()

# Leakage evaluation setup
honest_cols = [
    "imp_prev30",
    "days_with_impressions",
    "avg_position",
    "avg_daily_clicks",
]
leaky_col = "imp_last30"
label_col = "is_declining"

df_model = feature_frame.dropna(subset=honest_cols + [leaky_col, label_col])
X_tr_h, X_te_h, X_tr_l, X_te_l, y_tr, y_te = train_test_split(
    df_model[honest_cols],
    df_model[honest_cols + [leaky_col]],
    df_model[label_col],
    test_size=0.25,
    random_state=42,
    stratify=df_model[label_col],
)

# Fit honest vs leaky models
rf_h = RandomForestClassifier(random_state=42).fit(X_tr_h, y_tr)
rf_l = RandomForestClassifier(random_state=42).fit(X_tr_l, y_tr)

print(
    "Honest AP:",
    average_precision_score(y_te, rf_h.predict_proba(X_te_h)[:, 1]),
)
print(
    "Leaky AP:", average_precision_score(y_te, rf_l.predict_proba(X_te_l)[:, 1])
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest AP: 0.9983108845816722
Leaky AP: 0.999976306246098


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. **Unbalanced panel history:** Clients have different starting dates (`gsc_data_start`), making lookback windows non-uniform.
2. **GA4 zero-filling:** Missing GA4 periods are filled with `0` instead of `NULL`; filtering on `ga4_data_available IS TRUE` is mandatory.
3. **Current-window proxy label:** `is_declining` measures current decline rather than predicting true future drift.
4. **New page blindness:** Pages with zero initial impressions (`imp_prev30 = 0`) are excluded from training.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.